# import and data

In [4]:
# =============================================================================
# imports
# =============================================================================
import sqlite3
import pandas as pd
import sys, os; sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..","..")))
import utils as utils

# configuration
config = utils._load_config()

# nested shortcuts
db_cfg  = config.get("database", {})
dev_cfg = db_cfg.get("dev_data", {})

# top values from config
DB_PATH         = db_cfg.get("db_path")
TABLE_NAME_DEV  = dev_cfg.get("table_name_dev")
ROLLING_WINDOW  = dev_cfg.get("rolling_window")
target          = dev_cfg.get("target")

# dev params
open_time_from  = "2017-01-01 00:00:00"
open_time_to    = "2025-10-29 00:00:00"

# =============================================================================
# database connection
# =============================================================================
conn = sqlite3.connect(DB_PATH)

query = f"""
    SELECT t.*
    FROM {TABLE_NAME_DEV} t
    WHERE open_time BETWEEN '{open_time_from}' AND '{open_time_to}'
    ORDER BY open_time ASC
"""
df_dev = pd.read_sql_query(query, conn)
conn.close()

df_dev["open_time"] = pd.to_datetime(df_dev["open_time"], utc=True) 
df_dev.set_index("open_time", inplace=True)

display(df_dev.tail())

,close,rolling_max,ratio,spike_flag,feat_rsi_backward_14,feat_roc_14,feat_roc_140,feat_macd_diff,feat_sma_ratio_14,feat_sma_ratio_140,feat_bb_width_14,feat_bb_width_140
open_time,,,,,,,,,,,,
2025-10-28 09:45:00+00:00,557.6,558.1,1.000897,0,52.729870,0.107720,0.215672,0.266866,1.001912,1.000381,0.484893,1.762359
2025-10-28 09:46:00+00:00,557.6,558.1,1.000897,0,52.729870,0.107720,0.161667,0.276710,1.001835,1.000369,0.517660,1.761886
2025-10-28 09:47:00+00:00,557.7,558.1,1.000717,0,53.529260,0.287718,0.233645,0.279478,1.001809,1.000532,0.546854,1.760909
2025-10-28 09:48:00+00:00,558.1,558.1,1.000000,0,56.684686,0.233477,0.269493,0.296173,1.002360,1.001230,0.605864,1.760691
2025-10-28 09:49:00+00:00,557.4,557.4,1.000000,0,50.253809,0.215750,0.197735,0.249848,1.000949,0.999960,0.603681,1.759360


# model

In [5]:
# =============================================================================
# parameters for logistic elimination
# =============================================================================
import statsmodels.api as sm
from IPython.display import display

p_threshold = 0.01    # p-value cutoff for keeping a variable
max_iter = 100        # maximum elimination iterations

# =============================================================================
# prepare features list
# =============================================================================
# Expect df_dev to be available in the environment (DataFrame with feature cols and target)
features = [col for col in df_dev.columns if col.startswith("feat_")]

# =============================================================================
# CREATE df_sample BY FILTERING df_dev
# =============================================================================
# Drop rows with NA in any explanatory variable or the target
df_sample = df_dev.dropna(subset=features + [target])

# Exclude the most recent ROLLING_WINDOW minutes of observations
cutoff_time = df_sample.index.max() - pd.Timedelta(minutes=ROLLING_WINDOW)
df_sample   = df_sample.loc[df_sample.index < cutoff_time]

# =============================================================================
# LOGISTIC REGRESSION MODEL (iterative backward elimination)
# =============================================================================
remaining = features.copy()
removed   = []   # list of tuples (feature_name, p_value)
result    = None

for iteration in range(1, max_iter + 1):
    if not remaining:
        print("No features remaining to fit. Stopping.")
        break

    X = df_sample[remaining]
    X = sm.add_constant(X, has_constant='add')  # keep intercept
    y = df_sample[target]

    try:
        model = sm.Logit(y, X).fit(disp=False)
    except Exception as fit_err:
        print(f"Model fitting failed at iteration {iteration}: {fit_err}")
        # fallback: try a regularized fit to handle separation/numerical issues
        try:
            model = sm.Logit(y, X).fit_regularized(disp=False)
            print("Regularized fit succeeded as fallback.")
        except Exception as reg_err:
            print(f"Regularized fit also failed: {reg_err}. Stopping iteration.")
            break

    # p-values excluding the constant
    pvalues = model.pvalues.drop(labels=['const'], errors='ignore')

    if pvalues.empty:
        result = model
        print("No explanatory variables left after dropping constants.")
        break

    worst_var = pvalues.idxmax()
    worst_p = float(pvalues.max())

    print(f"Iteration {iteration}: {len(remaining)} features. Worst p-value = {worst_p:.6f} ({worst_var})")

    if worst_p <= p_threshold:
        result = model
        print(f"All remaining p-values <= {p_threshold}. Stopping elimination.")
        break

    # remove the worst variable and continue
    removed.append((worst_var, worst_p))
    remaining.remove(worst_var)

else:
    # reached max_iter without meeting the threshold
    print("Reached maximum iterations without satisfying p-value threshold.")
    try:
        X_final = sm.add_constant(df_sample[remaining], has_constant='add')
        result = sm.Logit(df_sample[target], X_final).fit(disp=False)
    except Exception as final_err:
        print("Final fit after max iterations failed:", final_err)
        result = None

# =============================================================================
# reporting results
# =============================================================================
print("\nRemoved features (in order):")
for name, pv in removed:
    print(f" - {name}: p = {pv:.6g}")

print("\nRemaining features:")
print(remaining)

if result is not None:
    try:
        display(result.summary().tables[1])
    except Exception:
        print(result.summary())
else:
    print("No final model available to display.")

Iteration 1: 8 features. Worst p-value = 0.294917 (feat_macd_diff)
Iteration 2: 7 features. Worst p-value = 0.372094 (feat_sma_ratio_140)
Iteration 3: 6 features. Worst p-value = 0.000000 (feat_roc_14)
All remaining p-values <= 0.01. Stopping elimination.

Removed features (in order):
 - feat_macd_diff: p = 0.294917
 - feat_sma_ratio_140: p = 0.372094

Remaining features:
['feat_rsi_backward_14', 'feat_roc_14', 'feat_roc_140', 'feat_sma_ratio_14', 'feat_bb_width_14', 'feat_bb_width_140']


,coef,std err,z,P>|z|,[0.025,0.975]
const,12.5981,1.135,11.098,0.000,10.373,14.823
feat_rsi_backward_14,0.0071,0.000,27.370,0.000,0.007,0.008
feat_roc_14,-0.0481,0.006,-7.788,0.000,-0.060,-0.036
feat_roc_140,0.0105,0.001,9.936,0.000,0.008,0.013
feat_sma_ratio_14,-15.9722,1.140,-14.015,0.000,-18.206,-13.739
feat_bb_width_14,0.6122,0.003,184.849,0.000,0.606,0.619
feat_bb_width_140,0.1799,0.001,170.549,0.000,0.178,0.182


# exports

In [12]:
import json
from pathlib import Path

# =============================================================================
# export only remaining feature names to JSON
# =============================================================================
output_path = Path("features.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(remaining, f, ensure_ascii=False, indent=4)

print(f"\n✅ Remaining feature nevek elmentve ide: {output_path.resolve()}")


# =============================================================================
# Slim save – adatmentes mentés, hogy a fájl kicsi maradjon
# =============================================================================

import os
import pickle

# Modellmentés célmappa és útvonal
model_path = "model.pkl"

# Előmelegítés: néhány statisztika cache-elése, hogy remove_data után is elérhető legyen
try:
    _ = model.summary()
except Exception:
    pass

# Elsődleges módszer: hivatalos statsmodels save() adatmentéssel
try:
    model.save(model_path, remove_data=True)
    print(f"✅ Slim model saved to: {model_path} (via save(remove_data=True))")

# Fallback: ha a save() remove_data paraméter nem támogatott
except Exception:
    print("⚠️ .save(remove_data=True) nem támogatott, fallback mentés indul...")
    try:
        model.remove_data()
    except Exception:
        pass
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"✅ Slim model saved to: {model_path} (via pickle fallback)")




✅ Remaining feature nevek elmentve ide: D:\repos\chronoquant\model_dev\logreg_base\features.json
✅ Slim model saved to: model.pkl (via save(remove_data=True))
